In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp04/ex_8.py ──
"""
Shared infrastructure for MLFP04 Exercise 8 — Deep Learning Foundations.

Contains: synthetic XOR data, synthetic chest-film-style triage images
(labels caused by drawn shapes), reusable training loops, gradient
monitoring and AUC helpers, ModelVisualizer output paths. Technique-specific code (model classes, per-file training
loops, scenario narratives) does NOT belong here — it lives per file.
"""

from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from kailash_ml import ModelVisualizer


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT — seeds, device, output dir
# ════════════════════════════════════════════════════════════════════════
setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

OUTPUT_DIR = Path("outputs") / "ex8_deep_learning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Shared hyperparameters
N_FEATS_XOR = 4
N_XOR_SAMPLES = 200
N_IMG_SAMPLES = 5000
IMG_SIZE = 64
N_CHANNELS = 1
N_CLASSES = 5
BATCH_SIZE = 64

# Kailash visualiser (used by every phase 4 block)
viz = ModelVisualizer()

# ════════════════════════════════════════════════════════════════════════
# DATA — XOR toy problem (Tasks 1-3)
# ════════════════════════════════════════════════════════════════════════


def make_xor_data(
    n_samples: int = N_XOR_SAMPLES, n_features: int = N_FEATS_XOR, seed: int = 42
) -> tuple[torch.Tensor, torch.Tensor, np.ndarray]:
    """Generate a synthetic XOR classification task.

    Label is XOR of the sign of features 0 and 1. Features 2..n-1 are noise.
    Returns (X_tensor, y_tensor, y_numpy) on CPU.
    """
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n_samples, n_features)).astype(np.float32)
    y = ((X[:, 0] > 0) ^ (X[:, 1] > 0)).astype(np.float32)
    X_t = torch.from_numpy(X)
    y_t = torch.from_numpy(y).unsqueeze(1)
    return X_t, y_t, y


# ════════════════════════════════════════════════════════════════════════
# DATA — Synthetic chest-film-style triage images (Tasks 4-10)
# ════════════════════════════════════════════════════════════════════════
# Scenario: chest-film triage at a Singapore public hospital. Real films
# are 512x512 DICOMs that cannot be shipped with a course, so this exercise
# draws SYNTHETIC 64x64 images in which each "finding" is a simple shape
# placed on a noisy background. The labels are caused by what is drawn, so
# a CNN can genuinely learn them — but these are NOT medical images, and
# nothing learned here says anything about real radiology.
#
#   pneumonia   -> a large, diffuse bright blob ("opacity")
#   effusion    -> a bright horizontal band across the base of the image
#   atelectasis -> a thin vertical bright streak
#   nodule      -> a small, sharp, very bright dot
#   normal      -> none of the four findings drawn
#
# Findings are independent (multi-label), each present in ~25% of images.

SG_HOSPITAL_CLASSES = [
    "pneumonia",
    "effusion",
    "atelectasis",
    "nodule",
    "normal",
]
FINDING_PREVALENCE = 0.25


def make_sg_imaging_data(
    n_samples: int = N_IMG_SAMPLES, seed: int = 42
) -> tuple[np.ndarray, np.ndarray]:
    """Return (X_images, y_labels) as float32 numpy arrays.

    X: (N, 1, 64, 64) — synthetic single-channel images: Gaussian background
       noise, a random per-image brightness offset, plus the drawn findings.
    y: (N, 5) — multi-label targets; columns follow SG_HOSPITAL_CLASSES and
       each finding column is 1 exactly when that shape was drawn.
    """
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:IMG_SIZE, 0:IMG_SIZE]
    X = rng.normal(0.0, 0.5, size=(n_samples, N_CHANNELS, IMG_SIZE, IMG_SIZE))
    X += rng.normal(0.0, 0.3, size=(n_samples, 1, 1, 1))  # exposure jitter
    present = rng.random((n_samples, 4)) < FINDING_PREVALENCE

    for i in range(n_samples):
        img = X[i, 0]
        if present[i, 0]:  # pneumonia: diffuse blob
            cy, cx = rng.uniform(16, 48, size=2)
            radius = rng.uniform(6, 10)
            img += 1.2 * np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * radius**2))
        if present[i, 1]:  # effusion: bright band at the base
            top = int(rng.integers(46, 56))
            img[top:, :] += 1.0
        if present[i, 2]:  # atelectasis: thin vertical streak
            col = int(rng.integers(8, 56))
            row0 = int(rng.integers(4, 24))
            length = int(rng.integers(20, 36))
            img[row0 : row0 + length, col : col + 2] += 1.5
        if present[i, 3]:  # nodule: small, sharp, bright dot
            cy, cx = rng.uniform(8, 56, size=2)
            radius = rng.uniform(2.0, 3.5)
            img += 2.5 * (((yy - cy) ** 2 + (xx - cx) ** 2) <= radius**2)

    y = np.zeros((n_samples, N_CLASSES), dtype=np.float32)
    y[:, :4] = present
    y[:, 4] = ~present.any(axis=1)
    return X.astype(np.float32), y


def build_sg_loaders(
    batch_size: int = BATCH_SIZE,
) -> tuple[DataLoader, DataLoader, np.ndarray, np.ndarray]:
    """Produce (train_loader, test_loader, X_test_np, y_test_np) for the CNN tasks."""
    X, y = make_sg_imaging_data()
    split = int(0.8 * len(X))
    X_tr, X_te = X[:split], X[split:]
    y_tr, y_te = y[:split], y[split:]

    train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
    test_ds = TensorDataset(torch.from_numpy(X_te), torch.from_numpy(y_te))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)
    return train_loader, test_loader, X_te, y_te


# ════════════════════════════════════════════════════════════════════════
# TRAINING UTILITIES
# ════════════════════════════════════════════════════════════════════════


def train_xor_net(
    net: nn.Module,
    X: torch.Tensor,
    y: torch.Tensor,
    optimiser: torch.optim.Optimizer,
    n_epochs: int = 100,
    criterion: nn.Module | None = None,
) -> list[float]:
    """Fit a small binary classifier to XOR data. Returns per-epoch loss."""
    crit = criterion or nn.BCEWithLogitsLoss()
    losses: list[float] = []
    for _ in range(n_epochs):
        optimiser.zero_grad()
        loss = crit(net(X), y)
        loss.backward()
        optimiser.step()
        losses.append(loss.item())
    return losses


def xor_accuracy(net: nn.Module, X: torch.Tensor, y_np: np.ndarray) -> float:
    """Binary accuracy on XOR data (threshold at 0.5)."""
    net.eval()
    with torch.no_grad():
        probs = torch.sigmoid(net(X)).numpy().flatten()
    return float(((probs > 0.5) == y_np).mean())


def grad_norm(model: nn.Module) -> float:
    """L2 norm of the concatenated gradient vector."""
    total = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total += p.grad.data.norm(2).item() ** 2
    return float(total**0.5)


def train_cnn_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimiser: torch.optim.Optimizer,
    criterion: nn.Module,
    clip_value: float | None = None,
) -> tuple[float, float]:
    """Train for one epoch on the synthetic triage-image loader.

    Returns (mean_loss, mean_grad_norm). If ``clip_value`` is set, the grad
    norm is measured pre-clipping and ``clip_grad_norm_`` is applied.
    """
    model.train()
    losses: list[float] = []
    grads: list[float] = []
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimiser.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward()
        grads.append(grad_norm(model))
        if clip_value is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_value)
        optimiser.step()
        losses.append(loss.item())
    return float(np.mean(losses)), float(np.mean(grads))


def eval_cnn(model: nn.Module, loader: DataLoader, criterion: nn.Module) -> float:
    """Return mean validation loss across the loader."""
    model.eval()
    losses: list[float] = []
    with torch.no_grad():
        for X_b, y_b in loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            losses.append(criterion(model(X_b), y_b).item())
    return float(np.mean(losses))


def eval_cnn_auc(
    model: nn.Module, X: np.ndarray, y: np.ndarray, batch_size: int = 256
) -> dict[str, float]:
    """Per-class ROC AUC on held-out images, plus their macro average.

    Loss values are hard to interpret on their own; AUC answers "does the
    model rank images WITH a finding above images WITHOUT it?" (0.5 =
    chance, 1.0 = perfect).
    """
    from sklearn.metrics import roc_auc_score

    model.eval()
    scores: list[np.ndarray] = []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            batch = torch.from_numpy(X[start : start + batch_size]).to(device)
            scores.append(torch.sigmoid(model(batch)).cpu().numpy())
    probs = np.concatenate(scores)
    aucs = {
        name: float(roc_auc_score(y[:, k], probs[:, k]))
        for k, name in enumerate(SG_HOSPITAL_CLASSES)
    }
    aucs["macro"] = float(np.mean(list(aucs.values())))
    return aucs


# ════════════════════════════════════════════════════════════════════════
# CNN BUILDING BLOCKS (reused across files 03, 04, 05)
# ════════════════════════════════════════════════════════════════════════


class ResBlock(nn.Module):
    """Residual block: skip connection preserves gradient flow."""

    def __init__(self, channels: int) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # noqa: D401
        residual = x
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return torch.relu(out + residual)


class TriageCNN(nn.Module):
    """CNN for multi-label Singapore hospital triage.

    Architecture: Conv32 -> ResBlock -> Conv64 -> ResBlock -> AdaptiveAvgPool
    -> Dropout -> Linear. Designed for the multi-label BCEWithLogitsLoss
    setup used throughout Exercise 8.
    """

    def __init__(self, n_classes: int = N_CLASSES, dropout_rate: float = 0.3) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            ResBlock(32),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            ResBlock(64),
            nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # noqa: D401
        return self.classifier(self.features(x))


def count_params(model: nn.Module) -> tuple[int, int]:
    """Return (total_params, trainable_params)."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# ════════════════════════════════════════════════════════════════════════
# DATA LOADER ENTRY POINT
# ════════════════════════════════════════════════════════════════════════
# We expose an MLFPDataLoader handle so student files have a single import
# path even though the images are generated on the fly. Real image datasets
# for CNN training and fine-tuning live in Module 5.
loader = MLFPDataLoader()




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 8.5: Regularisation, Clipping, Early Stop, ONNX Export
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Contrast baseline vs dropout vs batch-norm vs both
#   - Apply gradient clipping to prevent exploding gradients
#   - Implement early stopping on validation loss
#   - Export the trained CNN to ONNX via kailash-ml OnnxBridge
#
# PREREQUISITES: 04_optimisers_schedulers.py
#
# ESTIMATED TIME: ~45 min
#
# TASKS:
#   1. Theory — why training tricks compound
#   2. Build — four regularisation variants + EarlyStopping class
#   3. Train — full loop with clipping, scheduler, early stop
#   4. Visualise — val loss across variants + final gradient norms
#   5. Apply — edge deployment: shipping the CNN without a Python runtime
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import math

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

import onnxruntime as ort
from kailash_ml import OnnxBridge


print("\n" + "=" * 70)
print("  Regularisation, Clipping, Early Stopping, and ONNX Export")
print("=" * 70)



## THEORY — The regularisation stack


In [ ]:
# Regularisation is anything that limits what the network can memorise:
#   Dropout     — zero a random subset of activations at train time,
#                 forcing the network to distribute representation.
#   BatchNorm   — normalise activations per batch, which stabilises
#                 the optimisation landscape and acts as a mild
#                 regulariser by injecting batch noise.
#   Weight decay — shrink weights towards zero every step (AdamW).
#   Early stop  — stop training when val loss plateaus, using the
#                 training-time window itself as a regulariser.
#   Grad clip   — cap the gradient norm so a single pathological
#                 batch cannot knock the weights to infinity.
#
# Used together, they compound: dropout prevents co-adaptation, BN
# stabilises the landscape so clipping rarely fires, and early stopping
# picks the best checkpoint of the ones those tricks produced.

train_loader, test_loader, X_test_np, y_test_np = build_sg_loaders()
criterion = nn.BCEWithLogitsLoss()



## TASK 2 — BUILD: regularisation variants and EarlyStopping


A smaller variant of TriageCNN with configurable regularisation.


Monitor validation loss and stop when it plateaus.


In [ ]:
def build_variant(use_dropout: bool, use_bn: bool) -> nn.Module:
    layers: list[nn.Module] = [nn.Conv2d(1, 32, 3, padding=1)]
    if use_bn:
        layers.append(nn.BatchNorm2d(32))
    layers += [nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(32, 64, 3, padding=1)]
    if use_bn:
        layers.append(nn.BatchNorm2d(64))
    layers += [
        nn.ReLU(),
        nn.AdaptiveAvgPool2d(4),
        nn.Flatten(),
        nn.Linear(64 * 4 * 4, 64),
        nn.ReLU(),
    ]
    if use_dropout:
        layers.append(nn.Dropout(0.3))
    layers.append(nn.Linear(64, N_CLASSES))
    return nn.Sequential(*layers).to(device)


class EarlyStopping:

    def __init__(self, patience: int = 4, min_delta: float = 1e-3) -> None:
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float("inf")
        self.should_stop = False

    def step(self, val_loss: float) -> bool:
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop



## TASK 3 — TRAIN: regularisation grid + full pipeline with clipping + ES


In [ ]:
print("\n--- Regularisation grid (6 epochs each) ---")
variant_labels = [
    (False, False, "Baseline"),
    (True, False, "Dropout"),
    (False, True, "BatchNorm"),
    (True, True, "Dropout+BN"),
]

variant_val_histories: dict[str, list[float]] = {}
for use_do, use_bn, label in variant_labels:
    net = build_variant(use_do, use_bn)
    opt_ = optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    val_curve: list[float] = []
    for _ in range(6):
        train_cnn_one_epoch(net, train_loader, opt_, criterion)
        val_curve.append(eval_cnn(net, test_loader, criterion))
    variant_val_histories[label] = val_curve
    print(f"  {label:<12}: val {val_curve[0]:.4f} -> {val_curve[-1]:.4f}")



### Checkpoint A


In [ ]:
assert (
    variant_val_histories["Dropout+BN"][-1]
    <= variant_val_histories["Baseline"][-1] + 0.2
), "Task 3: Dropout+BN should not be dramatically worse than baseline"

# Full pipeline: TriageCNN + AdamW + cosine + clip + early stop
print("\n--- Full pipeline: TriageCNN + AdamW + cosine + clip + early stop ---")
model = TriageCNN(n_classes=N_CLASSES, dropout_rate=0.3).to(device)
optimiser = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

n_epochs = 12
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimiser, T_max=n_epochs, eta_min=1e-6
)
stopper = EarlyStopping(patience=4, min_delta=1e-3)

history = {"train_loss": [], "val_loss": [], "lr": [], "grad_norm": []}
stopped_epoch = n_epochs
for epoch in range(n_epochs):
    train_loss, grad = train_cnn_one_epoch(
        model, train_loader, optimiser, criterion, clip_value=1.0
    )
    val_loss = eval_cnn(model, test_loader, criterion)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["lr"].append(scheduler.get_last_lr()[0])
    history["grad_norm"].append(grad)
    scheduler.step()
    print(
        f"  Epoch {epoch + 1:>2}/{n_epochs}: "
        f"train={train_loss:.4f} val={val_loss:.4f} "
        f"lr={history['lr'][-1]:.6f} grad={grad:.3f}"
    )
    if stopper.step(val_loss):
        stopped_epoch = epoch + 1
        print(f"  -> early stop triggered at epoch {stopped_epoch}")
        break



### Checkpoint B


In [ ]:
assert stopper.best_loss < math.inf, "Task 3: early stopping must record a best loss"
assert all(
    g > 0 for g in history["grad_norm"]
), "Task 3: gradient norms must be positive"
print("\n[ok] Checkpoint passed — full training pipeline complete\n")



## TASK 4 — VISUALISE the regularisation grid and the final run


In [ ]:
fig_variants = viz.training_history(variant_val_histories, x_label="Epoch")
fig_variants.update_layout(title="Regularisation Variants — Validation BCE")
var_path = OUTPUT_DIR / "05_variants_val.html"
fig_variants.write_html(var_path)
print(f"[viz] Variants: {var_path}")

fig_full = viz.training_history(
    {
        "Train BCE": history["train_loss"],
        "Val BCE": history["val_loss"],
        "Grad Norm": history["grad_norm"],
    },
    x_label="Epoch",
)
fig_full.update_layout(title="Full Pipeline — Loss and Gradient Norm")
full_path = OUTPUT_DIR / "05_full_pipeline.html"
fig_full.write_html(full_path)
print(f"[viz] Full pipeline: {full_path}")



### (C) Dropout effect: side-by-side final val loss comparison


plotly 6.x rejects 8-digit #RRGGBBAA hex; emit rgba() instead.


In [ ]:
import plotly.graph_objects as go

variant_names = list(variant_val_histories.keys())
variant_final = [variant_val_histories[v][-1] for v in variant_names]
variant_initial = [variant_val_histories[v][0] for v in variant_names]
colors = ["#FECB52", "#636EFA", "#00CC96", "#AB63FA"]


def _rgba(hex_color: str, alpha: float) -> str:
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r}, {g}, {b}, {alpha})"


fig_dropout = go.Figure()
fig_dropout.add_trace(
    go.Bar(
        x=variant_names,
        y=variant_initial,
        name="Epoch 1 Val Loss",
        marker_color=[_rgba(c, 0.53) for c in colors],
        text=[f"{v:.4f}" for v in variant_initial],
        textposition="outside",
    )
)
fig_dropout.add_trace(
    go.Bar(
        x=variant_names,
        y=variant_final,
        name="Epoch 6 Val Loss",
        marker_color=colors,
        text=[f"{v:.4f}" for v in variant_final],
        textposition="outside",
    )
)
fig_dropout.update_layout(
    title="Regularisation Effect: Initial vs Final Validation Loss",
    xaxis_title="Variant",
    yaxis_title="Validation BCE Loss",
    barmode="group",
)
dropout_path = OUTPUT_DIR / "05_dropout_comparison.html"
fig_dropout.write_html(str(dropout_path))
print(f"[viz] Dropout comparison: {dropout_path}")



### (D) Train vs val loss with early stopping marker


In [ ]:
epochs_run = list(range(1, len(history["train_loss"]) + 1))
fig_es = go.Figure()
fig_es.add_trace(
    go.Scatter(
        x=epochs_run,
        y=history["train_loss"],
        mode="lines+markers",
        name="Train Loss",
        marker_color="#636EFA",
    )
)
fig_es.add_trace(
    go.Scatter(
        x=epochs_run,
        y=history["val_loss"],
        mode="lines+markers",
        name="Val Loss",
        marker_color="#EF553B",
    )
)
if stopped_epoch < n_epochs:
    fig_es.add_vline(
        x=stopped_epoch,
        line_dash="dash",
        line_color="green",
        annotation_text=f"Early Stop (epoch {stopped_epoch})",
        annotation_position="top left",
    )
# Mark the best val loss epoch
best_val_epoch = int(np.argmin(history["val_loss"])) + 1
best_val_loss = min(history["val_loss"])
fig_es.add_trace(
    go.Scatter(
        x=[best_val_epoch],
        y=[best_val_loss],
        mode="markers",
        marker=dict(size=14, color="green", symbol="star"),
        name=f"Best Val (epoch {best_val_epoch})",
    )
)
fig_es.update_layout(
    title="Training vs Validation Loss with Early Stopping",
    xaxis_title="Epoch",
    yaxis_title="BCE Loss",
)
es_path = OUTPUT_DIR / "05_early_stopping.html"
fig_es.write_html(str(es_path))
print(f"[viz] Early stopping: {es_path}")

# INTERPRETATION (computed from this run):
best_variant = min(variant_val_histories, key=lambda v: variant_val_histories[v][-1])
print("\nRegularisation grid — final validation BCE:")
for label in variant_val_histories:
    print(f"  {label:<12} {variant_val_histories[label][-1]:.4f}")
print(f"  Lowest final val loss this run: {best_variant}")
clip_epochs = sum(g > 1.0 for g in history["grad_norm"])
print(
    f"Full run: mean pre-clip gradient norm exceeded the clip threshold (1.0) "
    f"in {clip_epochs} of {len(history['grad_norm'])} epochs."
)
if clip_epochs:
    print(
        "  -> In those epochs the clipper rescaled at least the typical step "
        "back to norm 1.0, capping how far one batch can move the weights."
    )
else:
    print(
        "  -> The average step never needed clipping here; clipping is "
        "insurance against rare spikes, not something that fires constantly."
    )
final_aucs = eval_cnn_auc(model, X_test_np, y_test_np)
print(
    f"Best val loss {best_val_loss:.4f} at epoch {best_val_epoch}; "
    f"test macro AUC of the final model {final_aucs['macro']:.3f} "
    f"(per class: "
    + ", ".join(f"{n} {final_aucs[n]:.2f}" for n in SG_HOSPITAL_CLASSES)
    + ")"
)



## TASK 5 — APPLY: Edge Deployment Without a Python Runtime


In [ ]:
# SCENARIO (hypothetical): a Singapore team needs to run this triage
# model on small edge devices with limited RAM and no Python runtime.
# Shipping the PyTorch training stack (Python + torch, typically well
# over a gigabyte installed) is not an option. ONNX solves this: the
# model is serialised as a portable graph that ONNX Runtime — a far
# smaller native runtime with C/C++ APIs — can execute on many kinds of
# hardware.
#
# kailash-ml's OnnxBridge wraps the conversion: it checks compatibility
# up-front, exports torch modules (via torch.onnx.export with a dynamic
# batch dimension), and reports the outcome as a result object instead of
# raising, so a pipeline can decide what to do when export fails.

print("--- ONNX export via kailash-ml OnnxBridge ---")
# Step 1: Check compatibility before trying to export.
bridge = OnnxBridge()
compat = bridge.check_compatibility(model, framework="torch")
print(f"Compatible:  {compat.compatible}")
print(f"Confidence:  {compat.confidence}")

# Step 2: Export. torch exports trace the forward pass, so OnnxBridge
# needs a concrete sample input with the right shape.
onnx_output = OUTPUT_DIR / "triage_cnn.onnx"
model.eval()
sample_input = torch.from_numpy(X_test_np[:1]).to(device)
export_result = bridge.export(
    model, framework="torch", output_path=onnx_output, sample_input=sample_input
)
print(f"Export status: {export_result.onnx_status}")
if not export_result.success:
    raise RuntimeError(f"ONNX export failed: {export_result.error_message}")
model.eval()  # the trace runs in inference mode; keep it there for the check
# Recent torch exporters may store the weights in a side-car
# "<name>.onnx.data" file next to the graph — ship both files together.
onnx_files = [onnx_output] + sorted(OUTPUT_DIR.glob(onnx_output.name + ".data"))
total_kb = sum(f.stat().st_size for f in onnx_files) / 1024
print(f"Exported:    {', '.join(f.name for f in onnx_files)}")
print(f"Model size:  {total_kb:.1f} KB (graph + weights)")

# Step 3: Validate the exported graph against the PyTorch original with
# ONNX Runtime. A max-diff within tolerance means the edge runtime will
# produce the same predictions as the training runtime.
session = ort.InferenceSession(str(onnx_output))
input_name = session.get_inputs()[0].name
sample = X_test_np[:10]
with torch.no_grad():
    torch_out = model(torch.from_numpy(sample).to(device)).cpu().numpy()
onnx_out = session.run(None, {input_name: sample})[0]
max_diff = float(np.max(np.abs(torch_out - onnx_out)))
print(f"Max diff (torch vs onnx): {max_diff:.8f}")



### Checkpoint C


In [ ]:
assert compat.compatible, "Task 5: TriageCNN must be ONNX-compatible"
assert export_result.success, "Task 5: OnnxBridge export must succeed"
assert onnx_output.exists(), "Task 5: ONNX file must be written"
assert (
    max_diff < 1e-4
), f"Task 5: ONNX output must match PyTorch within 1e-4, got {max_diff}"
print("\n[ok] Checkpoint passed — model exported to ONNX and validated\n")

# BUSINESS IMPACT (illustrative): what you measured above is the part you
# can rely on — the exported file size and that ONNX Runtime reproduces
# the PyTorch outputs within 1e-4. The cost saving on a real fleet depends
# on the device, the runtime build and the batch size; benchmark latency
# and memory on the target hardware before quoting a number.
#
# LIMITATION: Not every PyTorch op has an ONNX equivalent. Dynamic
# control flow, custom autograd functions, and some advanced layers
# require rewriting for export. OnnxBridge reports compatibility up-front
# so you find this out before you ship, and export() returns a failed
# result (not an exception) when tracing breaks.



## REFLECTION


[x] Compared four regularisation variants on the same training budget
  [x] Built an EarlyStopping class with patience + min_delta
  [x] Ran a full TriageCNN pipeline with cosine decay, gradient clipping,
      and early stopping wired together
  [x] Exported the trained model to ONNX via kailash-ml OnnxBridge and
      validated the output against the PyTorch original
  [x] Planned an (illustrative) edge deployment that needs no Python runtime

  KEY INSIGHT: Regularisation tricks compound. You rarely need all of
  them — pick two that address the specific failure mode of your task
  (overfitting, exploding gradients, unstable training) and ship the
  model via ONNX.

  This completes Exercise 8 and Module 4. Module 5 takes these building
  blocks and applies them to real Fashion-MNIST, CIFAR-10, and transfer
  learning with pretrained backbones.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

# THE USML BRIDGE — COMPLETE



## Clustering / DR / association rules / matrix factorisation / neural
hidden layers — the bridge from "unsupervised ML without labels" to
"deep learning with labels" is now a continuous gradient. Hidden
layers learn representations automatically; everything upstream of
them was you choosing which representation to construct by hand.
